# Notebook 06 — NMF vs Sparse Autoencoder Recovery Comparison

**Purpose:** compare Notebook 04 (NMF baseline) and Notebook 05 (top-k sparse autoencoder) on one shared Mod30 recovery benchmark.

Core question:

```text
Does sparse learning behave differently than linear factorization
when recovering finite Mod30 residue tiles from continuous embeddings?
```

Shared metrics:

```text
coverage_fraction
mean_lane_purity
reconstruction_error
activation_sparsity
redundancy = total_activations / captured_lanes
effective_features = captured_lanes / total_components
dead_feature_fraction
redundant_feature_fraction
useful_feature_fraction
```

Paper-facing claim:

```text
Linear factorization can recover residue-class structure with dense activations,
while sparse autoencoders trade reconstruction and capacity efficiency for interpretable sparse latents.
```


## 0. Bulletproof setup

Run first.

In [ ]:

from pathlib import Path
import sys

def find_repo_root(start=None, marker="src"):
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    return None

REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "mod30-manifold-tiling"
    SRC_DIR = REPO_ROOT / "src"
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    (SRC_DIR / "__init__.py").write_text("", encoding="utf-8")
    (SRC_DIR / "mod30.py").write_text("""from math import gcd
MOD30 = 30
MOD30_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
def mod_index(n, mod): return n % mod
def mod_mask(n, residues, mod): return mod_index(n, mod) in residues
def generate_coprime_residues(mod): return [r for r in range(1, mod) if gcd(r, mod) == 1]
def mod30_index(n): return mod_index(n, MOD30)
def mod30_mask(n): return mod_mask(n, MOD30_RESIDUES, MOD30)
def residue_to_lane_index(residue): return MOD30_RESIDUES.index(residue) if residue in MOD30_RESIDUES else -1
def mod30_residues(n_max): return [n for n in range(2, n_max) if mod30_mask(n)]
""", encoding="utf-8")
    (SRC_DIR / "tiling_metrics.py").write_text("""import numpy as np
def lane_coverage(captured_residues, target_residues):
    captured, target = set(captured_residues), set(target_residues)
    hit, missed = captured & target, target - captured
    return {"target_lanes": len(target), "captured_lanes": len(hit), "missed_lanes": len(missed),
            "coverage_fraction": len(hit)/len(target) if target else 0.0,
            "captured_residues": sorted(hit), "missed_residues": sorted(missed)}
def reconstruction_error(X, X_hat):
    denom = np.linalg.norm(X)
    return 0.0 if denom == 0 else float(np.linalg.norm(X - X_hat) / denom)
def activation_sparsity(A, eps=1e-6): return float(np.mean(A <= eps))
def feature_lane_alignment(A, lane_labels, residues):
    align = np.zeros((A.shape[1], len(residues)))
    for j in range(A.shape[1]):
        total = A[:, j].sum()
        if total <= 0: continue
        for k, r in enumerate(residues):
            mask = lane_labels == r
            align[j, k] = A[mask, j].sum() / total
    return align
def lane_purity_from_alignment(align): return [] if align.size == 0 else align.max(axis=1).tolist()
def recovered_residues_from_alignment(align, residues, threshold=0.45):
    recovered = []
    for row in align:
        if row.max() >= threshold:
            recovered.append(residues[int(row.argmax())])
    return sorted(set(recovered))
def feature_usage_diagnostics(Z, align, useful_threshold=0.45, activity_eps=1e-6):
    feature_activity = np.sum(Z > activity_eps, axis=0)
    active_mask = feature_activity > 0
    dead_mask = ~active_mask
    purity = align.max(axis=1) if align.size else np.zeros(Z.shape[1])
    useful_mask = active_mask & (purity >= useful_threshold)
    redundant_mask = active_mask & (purity < useful_threshold)
    latent_dim = Z.shape[1]
    return {"feature_activity": feature_activity, "feature_purity": purity,
            "active_features": int(active_mask.sum()), "dead_features": int(dead_mask.sum()),
            "useful_features": int(useful_mask.sum()), "redundant_features": int(redundant_mask.sum()),
            "active_feature_fraction": float(active_mask.sum()/latent_dim) if latent_dim else 0.0,
            "dead_feature_fraction": float(dead_mask.sum()/latent_dim) if latent_dim else 0.0,
            "useful_feature_fraction": float(useful_mask.sum()/latent_dim) if latent_dim else 0.0,
            "redundant_feature_fraction": float(redundant_mask.sum()/latent_dim) if latent_dim else 0.0}
""", encoding="utf-8")
    (SRC_DIR / "plots.py").write_text("""import matplotlib.pyplot as plt
def save_current(path, dpi=180):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    return path
""", encoding="utf-8")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"
for d in [FIGURES_DIR, DATA_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("src exists:", (REPO_ROOT / "src").exists())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise ImportError("This notebook needs PyTorch. In Colab, Runtime usually includes torch by default.") from e

from src.mod30 import MOD30_RESIDUES, mod30_index, mod30_mask, residue_to_lane_index
from src.tiling_metrics import (
    lane_coverage,
    reconstruction_error,
    activation_sparsity,
    feature_lane_alignment,
    lane_purity_from_alignment,
    recovered_residues_from_alignment,
    feature_usage_diagnostics,
)
from src.plots import save_current

SEED = 9423
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("Persisting Mod30 lanes:", MOD30_RESIDUES)


## 1. Build shared Mod30 dataset and continuous embedding

In [ ]:
n_min = 1
n_max = 1500
values = np.arange(n_min, n_max + 1)

df = pd.DataFrame({
    "n": values,
    "mod30_residue": [mod30_index(int(n)) for n in values],
})
df["inside_mod30_gate"] = df["n"].apply(lambda n: mod30_mask(int(n)))
df["lane_index"] = df["mod30_residue"].apply(residue_to_lane_index)

gated_df = df[df["inside_mod30_gate"]].copy().reset_index(drop=True)

RNG = np.random.default_rng(SEED)

def build_continuous_embedding(gated_df, noise_scale=0.04, lane_strength=1.0):
    residues = gated_df["mod30_residue"].to_numpy()
    lane_idx = gated_df["lane_index"].to_numpy()

    theta = 2 * np.pi * residues / 30.0

    circle = np.column_stack([
        0.5 + 0.5 * np.cos(theta),
        0.5 + 0.5 * np.sin(theta),
    ])

    lane_channels = np.zeros((len(gated_df), len(MOD30_RESIDUES)))
    lane_channels[np.arange(len(gated_df)), lane_idx] = lane_strength

    for i in range(len(gated_df)):
        k = lane_idx[i]
        lane_channels[i, (k - 1) % len(MOD30_RESIDUES)] += 0.18
        lane_channels[i, (k + 1) % len(MOD30_RESIDUES)] += 0.18

    noise = np.abs(RNG.normal(loc=0.0, scale=noise_scale, size=(len(gated_df), 4)))

    X = np.column_stack([circle, lane_channels, noise])
    X = MinMaxScaler().fit_transform(X)
    return X.astype(np.float32)

X = build_continuous_embedding(gated_df)
X_tensor = torch.tensor(X, dtype=torch.float32, device=device)

print("X:", X.shape)
gated_df.head()


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=gated_df["lane_index"], s=10)
plt.xlabel("embedding dim 0")
plt.ylabel("embedding dim 1")
plt.title("Shared Mod30 continuous embedding")
save_current(FIGURES_DIR / "35_shared_mod30_embedding.png")
plt.show()


## 2. Shared evaluation helper

This evaluates either:

```text
NMF activations W
SAE activations Z
```

using identical recovery logic.


In [ ]:
USEFUL_THRESHOLD = 0.35

def evaluate_activations(model_name, setting, A, X_hat, total_components):
    align = feature_lane_alignment(
        A,
        gated_df["mod30_residue"].to_numpy(),
        MOD30_RESIDUES,
    )

    purities = lane_purity_from_alignment(align)
    recovered = recovered_residues_from_alignment(align, MOD30_RESIDUES, threshold=USEFUL_THRESHOLD)
    cov = lane_coverage(recovered, MOD30_RESIDUES)

    total_activations = float((A > 1e-6).sum())
    redundancy = total_activations / cov["captured_lanes"] if cov["captured_lanes"] else np.inf
    effective_features = cov["captured_lanes"] / total_components if total_components else 0.0

    usage = feature_usage_diagnostics(
        A,
        align,
        useful_threshold=USEFUL_THRESHOLD,
        activity_eps=1e-6,
    )

    return {
        "model": model_name,
        "setting": setting,
        "total_components": total_components,
        "reconstruction_error": reconstruction_error(X, X_hat),
        "activation_sparsity": activation_sparsity(A, eps=1e-6),
        "mean_lane_purity": float(np.mean(purities)) if purities else 0.0,
        "max_lane_purity": float(np.max(purities)) if purities else 0.0,
        "captured_lanes": cov["captured_lanes"],
        "coverage_fraction": cov["coverage_fraction"],
        "total_activations": total_activations,
        "redundancy": redundancy,
        "effective_features": effective_features,
        "active_features": usage["active_features"],
        "dead_features": usage["dead_features"],
        "useful_features": usage["useful_features"],
        "redundant_features": usage["redundant_features"],
        "active_feature_fraction": usage["active_feature_fraction"],
        "dead_feature_fraction": usage["dead_feature_fraction"],
        "useful_feature_fraction": usage["useful_feature_fraction"],
        "redundant_feature_fraction": usage["redundant_feature_fraction"],
        "recovered_residues": recovered,
        "missed_residues": cov["missed_residues"],
        "alignment": align,
        "activations": A,
        "X_hat": X_hat,
    }


## 3. Run NMF grid

In [ ]:
nmf_grid = [1, 4, 8, 12]
nmf_results = {}
nmf_rows = []

for k in nmf_grid:
    nmf = NMF(
        n_components=k,
        init="nndsvda",
        random_state=SEED,
        max_iter=1500,
        l1_ratio=0.65,
        alpha_W=0.002,
        alpha_H=0.002,
    )
    W = nmf.fit_transform(X)
    H = nmf.components_
    X_hat = W @ H

    setting = f"NMF-{k}"
    metric = evaluate_activations("NMF", setting, W, X_hat, total_components=k)
    nmf_results[k] = {"model": nmf, "W": W, "H": H, "metrics": metric}
    nmf_rows.append({k2: v for k2, v in metric.items() if k2 not in ["alignment", "activations", "X_hat"]})

nmf_metrics_df = pd.DataFrame(nmf_rows)
nmf_metrics_df


## 4. Define and run SAE grid

In [ ]:
def topk_mask(z, k):
    if k >= z.shape[1]:
        return z
    values, indices = torch.topk(z, k=k, dim=1)
    mask = torch.zeros_like(z)
    mask.scatter_(1, indices, 1.0)
    return z * mask

class TopKSparseAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, top_k):
        super().__init__()
        self.encoder = nn.Linear(input_dim, latent_dim)
        self.decoder = nn.Linear(latent_dim, input_dim, bias=True)
        self.top_k = top_k

    def forward(self, x):
        z_dense = F.relu(self.encoder(x))
        z_sparse = topk_mask(z_dense, self.top_k)
        x_hat = self.decoder(z_sparse)
        return x_hat, z_sparse, z_dense

def train_sae(latent_dim=8, top_k=1, epochs=1200, lr=1e-2, l1_weight=1e-4):
    torch.manual_seed(SEED + latent_dim * 100 + top_k)
    model = TopKSparseAutoencoder(X.shape[1], latent_dim, top_k).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        x_hat, z_sparse, z_dense = model(X_tensor)
        recon_loss = F.mse_loss(x_hat, X_tensor)
        l1_loss = z_dense.abs().mean()
        loss = recon_loss + l1_weight * l1_loss
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

    with torch.no_grad():
        x_hat, z_sparse, z_dense = model(X_tensor)

    return {
        "model": model,
        "losses": losses,
        "X_hat": x_hat.detach().cpu().numpy(),
        "Z": z_sparse.detach().cpu().numpy(),
        "Z_dense": z_dense.detach().cpu().numpy(),
    }


In [ ]:
sae_grid = [
    {"latent_dim": 1, "top_k": 1},
    {"latent_dim": 4, "top_k": 1},
    {"latent_dim": 8, "top_k": 1},
    {"latent_dim": 8, "top_k": 2},
    {"latent_dim": 12, "top_k": 1},
    {"latent_dim": 12, "top_k": 2},
]

sae_results = {}
sae_rows = []

for cfg in sae_grid:
    latent_dim = cfg["latent_dim"]
    top_k = cfg["top_k"]
    result = train_sae(latent_dim=latent_dim, top_k=top_k)

    setting = f"SAE-L{latent_dim}-k{top_k}"
    metric = evaluate_activations("SAE", setting, result["Z"], result["X_hat"], total_components=latent_dim)

    key = (latent_dim, top_k)
    sae_results[key] = {**result, "metrics": metric}
    sae_rows.append({k2: v for k2, v in metric.items() if k2 not in ["alignment", "activations", "X_hat"]})

sae_metrics_df = pd.DataFrame(sae_rows)
sae_metrics_df


## 5. Combined metrics table

In [ ]:
combined_df = pd.concat([nmf_metrics_df, sae_metrics_df], ignore_index=True)
combined_df.to_csv(DATA_DIR / "06_nmf_vs_sae_combined_metrics.csv", index=False)
combined_df


## 6. Coverage vs reconstruction comparison

In [ ]:
plt.figure(figsize=(8, 5.5))

for model_name, g in combined_df.groupby("model"):
    plt.scatter(g["coverage_fraction"], 1 - g["reconstruction_error"], s=120, label=model_name)
    for _, row in g.iterrows():
        plt.annotate(row["setting"], (row["coverage_fraction"], 1 - row["reconstruction_error"]),
                     textcoords="offset points", xytext=(6, 6), ha="left", fontsize=8)

plt.xlabel("coverage fraction")
plt.ylabel("1 - reconstruction error")
plt.title("NMF vs SAE: coverage vs reconstruction")
plt.legend()
save_current(FIGURES_DIR / "36_nmf_vs_sae_coverage_vs_reconstruction.png")
plt.show()


## 7. Coverage vs sparsity comparison

In [ ]:
plt.figure(figsize=(8, 5.5))

for model_name, g in combined_df.groupby("model"):
    plt.scatter(g["activation_sparsity"], g["coverage_fraction"], s=120, label=model_name)
    for _, row in g.iterrows():
        plt.annotate(row["setting"], (row["activation_sparsity"], row["coverage_fraction"]),
                     textcoords="offset points", xytext=(6, 6), ha="left", fontsize=8)

plt.xlabel("activation sparsity")
plt.ylabel("coverage fraction")
plt.title("NMF vs SAE: sparse activations vs lane coverage")
plt.legend()
save_current(FIGURES_DIR / "37_nmf_vs_sae_sparsity_vs_coverage.png")
plt.show()


## 8. Effective features and dead capacity

In [ ]:
plot_df = combined_df.copy()
x = np.arange(len(plot_df))
width = 0.18

plt.figure(figsize=(14, 6))
plt.bar(x - 1.5*width, plot_df["coverage_fraction"], width, label="coverage")
plt.bar(x - 0.5*width, plot_df["effective_features"], width, label="effective features")
plt.bar(x + 0.5*width, plot_df["dead_feature_fraction"], width, label="dead fraction")
plt.bar(x + 1.5*width, plot_df["redundant_feature_fraction"], width, label="redundant fraction")

plt.xticks(x, plot_df["setting"], rotation=30, ha="right")
plt.ylabel("metric value")
plt.title("NMF vs SAE: coverage, effective features, dead/redundant capacity")
plt.legend()
save_current(FIGURES_DIR / "38_nmf_vs_sae_effective_dead_redundant.png")
plt.show()


## 9. Redundancy vs coverage

In [ ]:
plt.figure(figsize=(8, 5.5))

for model_name, g in combined_df.groupby("model"):
    plt.scatter(g["redundancy"], g["coverage_fraction"], s=120, label=model_name)
    for _, row in g.iterrows():
        plt.annotate(row["setting"], (row["redundancy"], row["coverage_fraction"]),
                     textcoords="offset points", xytext=(6, 6), ha="left", fontsize=8)

plt.xlabel("redundancy = total activations / captured lanes")
plt.ylabel("coverage fraction")
plt.title("NMF vs SAE: redundancy vs coverage")
plt.legend()
save_current(FIGURES_DIR / "39_nmf_vs_sae_redundancy_vs_coverage.png")
plt.show()


## 10. Alignment matrix comparison at matched capacity

In [ ]:
# NMF 8 components vs SAE L8-k1
nmf_align = nmf_results[8]["metrics"]["alignment"]
sae_align = sae_results[(8, 1)]["metrics"]["alignment"]

plt.figure(figsize=(8, 5))
plt.imshow(nmf_align, aspect="auto", interpolation="nearest")
plt.xticks(range(len(MOD30_RESIDUES)), [f"r{r}" for r in MOD30_RESIDUES])
plt.yticks(range(8), [f"NMF{j}" for j in range(8)])
plt.xlabel("true residue lane")
plt.ylabel("component")
plt.title("NMF feature-to-lane alignment: 8 components")
plt.colorbar(label="fraction of activation")
save_current(FIGURES_DIR / "40_nmf_alignment_8_components.png")
plt.show()

plt.figure(figsize=(8, 5))
plt.imshow(sae_align, aspect="auto", interpolation="nearest")
plt.xticks(range(len(MOD30_RESIDUES)), [f"r{r}" for r in MOD30_RESIDUES])
plt.yticks(range(8), [f"SAE{j}" for j in range(8)])
plt.xlabel("true residue lane")
plt.ylabel("latent")
plt.title("SAE feature-to-lane alignment: L8-k1")
plt.colorbar(label="fraction of activation")
save_current(FIGURES_DIR / "41_sae_alignment_L8_k1.png")
plt.show()


## 11. Activation matrix comparison at matched capacity

In [ ]:
W_nmf = nmf_results[8]["W"]
Z_sae = sae_results[(8, 1)]["Z"]

plt.figure(figsize=(12, 4.8))
plt.imshow(W_nmf[:240].T, aspect="auto", interpolation="nearest")
plt.xlabel("sample index")
plt.ylabel("NMF component")
plt.title("NMF activation matrix: 8 components")
save_current(FIGURES_DIR / "42_nmf_activation_matrix_8.png")
plt.show()

plt.figure(figsize=(12, 4.8))
plt.imshow(Z_sae[:240].T, aspect="auto", interpolation="nearest")
plt.xlabel("sample index")
plt.ylabel("SAE latent")
plt.title("SAE activation matrix: L8-k1")
save_current(FIGURES_DIR / "43_sae_activation_matrix_L8_k1.png")
plt.show()


## 12. Best setting by model

In [ ]:
# Simple score: prioritize coverage and reconstruction, penalize dead/redundant fractions.
score_df = combined_df.copy()
score_df["score"] = (
    score_df["coverage_fraction"]
    + (1 - score_df["reconstruction_error"])
    + score_df["mean_lane_purity"]
    - 0.25 * score_df["dead_feature_fraction"]
    - 0.25 * score_df["redundant_feature_fraction"]
)

best_by_model = (
    score_df.sort_values("score", ascending=False)
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

best_by_model.to_csv(DATA_DIR / "06_best_settings_by_model.csv", index=False)
best_by_model


## 13. Interpretation

Notebook 06 turns the separate NMF and SAE experiments into one shared comparison.

Expected paper-facing pattern:

```text
NMF can give strong dense reconstruction and smooth factor recovery.
SAE produces explicitly sparse latents, but may leave capacity dead or redundant.
```

Main claim:

```text
Sparse interpretability is not free: sparsity improves legibility while introducing capacity-allocation failures that dense factorization does not expose in the same way.
```


## 14. Save compact summary

In [ ]:
summary_md = f"""# Notebook 06 Summary — NMF vs SAE Recovery Comparison

Notebook 06 compares NMF and top-k sparse autoencoder recovery on the same Mod30 continuous embedding benchmark.

## Combined metrics

{combined_df.to_markdown(index=False)}

## Best settings by model

{best_by_model.to_markdown(index=False)}

## Interpretation

NMF and SAE recover Mod30 residue-tile structure differently.

- NMF: dense linear factorization, smooth recovery, no explicit top-k sparsity.
- SAE: sparse latent activations, useful interpretability, but dead and redundant capacity can appear.

## Generated figures

- `figures/35_shared_mod30_embedding.png`
- `figures/36_nmf_vs_sae_coverage_vs_reconstruction.png`
- `figures/37_nmf_vs_sae_sparsity_vs_coverage.png`
- `figures/38_nmf_vs_sae_effective_dead_redundant.png`
- `figures/39_nmf_vs_sae_redundancy_vs_coverage.png`
- `figures/40_nmf_alignment_8_components.png`
- `figures/41_sae_alignment_L8_k1.png`
- `figures/42_nmf_activation_matrix_8.png`
- `figures/43_sae_activation_matrix_L8_k1.png`

## Generated data

- `data/06_nmf_vs_sae_combined_metrics.csv`
- `data/06_best_settings_by_model.csv`
"""

summary_path = OUTPUTS_DIR / "06_nmf_vs_sae_recovery_comparison_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)


## 15. Optional: zip-download pattern

Uncomment in Colab to download figures, data, and output summaries.


In [ ]:
# Optional zip-download pattern:
#
# import shutil
#
# bundle_name = "notebook_06_nmf_vs_sae_recovery_comparison_outputs"
# bundle_base = REPO_ROOT / bundle_name
# bundle_zip = REPO_ROOT / f"{bundle_name}.zip"
#
# if bundle_base.exists():
#     shutil.rmtree(bundle_base)
#
# bundle_base.mkdir(parents=True, exist_ok=True)
#
# for folder_name in ["figures", "data", "outputs"]:
#     src_folder = REPO_ROOT / folder_name
#     dst_folder = bundle_base / folder_name
#     if src_folder.exists():
#         shutil.copytree(src_folder, dst_folder)
#
# shutil.make_archive(str(bundle_base), "zip", bundle_base)
# print("Created:", bundle_zip)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(str(bundle_zip))


## 16. Recommended next step

This completes the first experimental arc.

Recommended next files:

```text
docs/additional_notebooks.md
paper/paper.md
paper/mod30_manifold_tiling.tex
```
